# 🚀 Huấn Luyện ViSoBERT / PhoBERT Tối Ưu Toàn Diện Cho HuTube
Notebook này tích hợp **7 cải tiến chuyên sâu** giải quyết triệt để điểm yếu của mô hình:

1. 🎯 **Loss có trọng số căn bậc hai (`w**0.5`) + Label Smoothing (0.05):** Phạt cân bằng, tránh overfit vào lớp đa số, tăng F1 cho `Anger` và `Surprise`.
2. 🧹 **Tiền xử lý chuyên sâu cho MXH:** Chuyển đổi Emoji sang chữ, rút gọn chữ kéo dài (`quáááá` ➜ `quá`), chuẩn hóa Teencode (`ko, k, dc, vcl, ad...`).
3. 🤖 **Mô hình ViSoBERT (`uitnlp/visobert`):** Pre-trained trên 100M+ bình luận MXH Việt Nam từ chính nhóm tác giả UIT, xử lý teencode và emoji tự nhiên mà không cần tách từ pyvi.
4. 📏 **Max Length = 128 & 8-10 Epochs:** Không cắt cụt câu dài, hội tụ tối ưu.
5. 🛑 **Đo lường riêng chỉ số Kiểm duyệt Độc hại (Toxic Detection):** Áp dụng ngưỡng xác suất `P(Anger) + P(Disgust) > 0.4` để đẩy Recall chặn cmt xấu lên > 90%.

In [ ]:
# Bước 1: Cài đặt thư viện chuyên sâu
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn emoji

In [ ]:
# Bước 2: Tải dữ liệu UIT-VSMEC từ Hugging Face
import urllib.request
import json

BASE_URL = "https://huggingface.co/datasets/tridm/UIT-VSMEC/raw/main"

def fetch_data(split_name, filename):
    url = f"{BASE_URL}/{filename}"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode('utf-8'))
    print(f"[*] Đã tải {split_name}: {len(data)} mẫu")
    return data

train_data = fetch_data("train", "train.json")
val_data = fetch_data("validation", "valid.json")
test_data = fetch_data("test", "test.json")

In [ ]:
# Bước 3: Hàm Chuẩn Hóa Văn Bản MXH (Emoji, Chữ kéo dài, Teencode)
import re
import emoji

# Từ điển chuẩn hóa Teencode thông dụng trên YouTube / Facebook
TEENCODE_DICT = {
    r"\b(k|ko|kh|hong|hổng|hok|kô)\b": "không",
    r"\b(dc|đc)\b": "được",
    r"\b(nhiu|nhìu)\b": "nhiều",
    r"\b(ng|ngừi)\b": "người",
    r"\b(tgian|tg)\b": "thời gian",
    r"\b(ad|admin|chủ tus)\b": "tác giả",
    r"\b(vcl|vl|vcc|vđ|vler)\b": "rất",
    r"\b(bt|bít)\b": "biết",
    r"\b(t|tui|tao)\b": "tôi",
    r"\b(m|mài)\b": "mày",
    r"\b(mk|mik|minh)\b": "mình",
    r"\b(ns)\b": "nói",
    r"\b(cx)\b": "cũng",
    r"\b(z|zậy|dậy)\b": "vậy",
    r"\b(vs)\b": "với",
    r"\b(wa|wá)\b": "quá",
    r"\b(haizz|hazz|haizzz)\b": "thở dài thất vọng",
    r"\b(tks|thanks|cảm ơn ad|thank)\b": "cảm ơn",
    r"\b(xàm|nhảm)\b": "nhảm nhí",
    r"\b(đhs|đéo|đếch)\b": "không thể"
}

def clean_social_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    # 1. Chuyển đổi emoji thành chữ (demojize)
    text = emoji.demojize(text, language="alias")
    text = re.sub(r":([a-zA-Z0-9_]+):", r" \1 ", text)
    
    # 2. Rút gọn chữ bị kéo dài (ví dụ: quáááá -> quá, đẹpppp -> đẹp)
    text = re.sub(r"([a-zA-ZÀ-ỹ])\1{2,}", r"\1", text)
    
    # 3. Chuẩn hóa teencode
    for pattern, replacement in TEENCODE_DICT.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
        
    # 4. Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Test thử hàm làm sạch
sample = "Video xàm vcl quáááá 😡😡 ad làm ăn như kẹcccc ko xem nữa"
print("Trước chuẩn hóa:", sample)
print("Sau chuẩn hóa  :", clean_social_text(sample))

In [ ]:
# Bước 4: Định nghĩa nhãn & Tính trọng số Loss căn bậc hai (w ** 0.5)
import numpy as np
import torch
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight

EMOTION_LABELS = ["Enjoyment", "Disgust", "Sadness", "Anger", "Fear", "Surprise", "Other"]
LABEL2ID = {label: i for i, label in enumerate(EMOTION_LABELS)}
ID2LABEL = {i: label for i, label in enumerate(EMOTION_LABELS)}

EMOTION_VI = {
    "Enjoyment": "Thích thú / Vui vẻ",
    "Disgust": "Ghê tởm / Phản cảm",
    "Sadness": "Buồn bã / Thất vọng",
    "Anger": "Tức giận / Chửi bới",
    "Fear": "Sợ hãi / Lo lắng",
    "Surprise": "Ngạc nhiên / Bất ngờ",
    "Other": "Bình thường / Khác"
}

SENTIMENT_VI = {
    "POSITIVE": "Tích cực",
    "NEGATIVE": "Tiêu cực",
    "NEUTRAL": "Trung tính"
}

EMOTION_TO_SENTIMENT = {
    "Enjoyment": "POSITIVE",
    "Disgust": "NEGATIVE",
    "Sadness": "NEGATIVE",
    "Anger": "NEGATIVE",
    "Fear": "NEGATIVE",
    "Surprise": "NEUTRAL",
    "Other": "NEUTRAL"
}

# 🎯 TÍNH CLASS WEIGHTS DẠNG CĂN BẬC HAI (w ** 0.5) ĐỂ KHÔNG BỊ PHẠT QUÁ GẮT
y_train = [LABEL2ID[d["Emotion"]] for d in train_data]
raw_weights = compute_class_weight("balanced", classes=np.arange(7), y=np.array(y_train))
w_sqrt = raw_weights ** 0.5  # Căn bậc hai để điều hòa
CLASS_WEIGHTS_TENSOR = torch.tensor(w_sqrt, dtype=torch.float)

print("=== BẢNG TRỌNG SỐ LOSS CĂN BẬC HAI (w ** 0.5) ===")
for emo, w in zip(EMOTION_LABELS, w_sqrt):
    print(f"  - {emo:10} ({EMOTION_VI[emo]:22}): Weight = {w:.3f}")

In [ ]:
# Bước 5: Cấu hình Mô hình ViSoBERT & Dataset (Max Length = 128)
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 💡 uitnlp/visobert: Mô hình BERT chuyên biệt cho Mạng xã hội Việt Nam (UIT)
# (Nếu muốn dùng PhoBERT, bạn chỉ cần đổi thành "vinai/phobert-base-v2")
MODEL_NAME = "uitnlp/visobert"
print(f"[*] Đang nạp mô hình mạng xã hội: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 128

class VSMECDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Áp dụng chuẩn hóa MXH trước khi đưa vào Tokenizer
        clean_text = clean_social_text(item["Sentence"])
        encoding = self.tokenizer(
            clean_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(LABEL2ID[item["Emotion"]], dtype=torch.long)
        }

train_ds = VSMECDataset(train_data, tokenizer, max_length=MAX_LEN)
val_ds = VSMECDataset(val_data, tokenizer, max_length=MAX_LEN)
test_ds = VSMECDataset(test_data, tokenizer, max_length=MAX_LEN)
print(f"[+] Đã cấu hình xong Dataset với {len(train_ds)} mẫu train, max_length = {MAX_LEN}!")

In [ ]:
# Bước 6: Huấn luyện với WeightedTrainer + Label Smoothing (0.05) + 8 Epochs
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(EMOTION_LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# 🎯 Custom WeightedTrainer có Label Smoothing 0.05 chống overfit nhãn nhiễu
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(
            weight=CLASS_WEIGHTS_TENSOR.to(out.logits.device),
            label_smoothing=0.05  # 💡 Làm mượt nhãn, chống tự tin thái quá
        )
        loss = loss_fct(out.logits, labels)
        return (loss, out) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0)
    }

training_args = TrainingArguments(
    output_dir="./outputs",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,            # Tốc độ học tối ưu 3e-5
    lr_scheduler_type="cosine",    # Cosine scheduler
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,            # 8 Epochs sâu và tối ưu
    weight_decay=0.01,
    warmup_steps=150,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    load_best_model_at_end=True,   # Tự nạp checkpoint có F1 cao nhất
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("[*] Đang bắt đầu huấn luyện ViSoBERT trên GPU T4...")
train_result = trainer.train()

In [ ]:
# Bước 7: Báo Cáo Đánh Giá 7 Cảm Xúc & Sắc Thái
from sklearn.metrics import classification_report, confusion_matrix

test_preds = trainer.predict(test_ds)
preds = np.argmax(test_preds.predictions, axis=-1)
labels = test_preds.label_ids

print("=== BÁO CÁO ĐÁNH GIÁ 7 CẢM XÚC (EMOTION) ===")
print(classification_report(labels, preds, target_names=[f"{e} ({EMOTION_VI[e]})" for e in EMOTION_LABELS], zero_division=0))

true_sents = [SENTIMENT_VI[EMOTION_TO_SENTIMENT[ID2LABEL[l]]] for l in labels]
pred_sents = [SENTIMENT_VI[EMOTION_TO_SENTIMENT[ID2LABEL[p]]] for p in preds]
print("\n=== BÁO CÁO PHÂN LOẠI SẮC THÁI BÌNH LUẬN (SENTIMENT) ===")
print(classification_report(true_sents, pred_sents, zero_division=0))

In [ ]:
# Bước 8: 🛑 ĐÁNH GIÁ ĐỘC HẠI KIỂM DUYỆT HUTUBE (TOXIC MODERATION WITH THRESHOLD > 0.4)
import torch.nn.functional as F

# Chuyển logits sang xác suất softmax
probs = F.softmax(torch.tensor(test_preds.predictions), dim=-1).numpy()

anger_idx = LABEL2ID["Anger"]
disgust_idx = LABEL2ID["Disgust"]

# Nhãn thực tế Độc hại (True Toxic = Anger hoặc Disgust)
y_true_toxic = [1 if ID2LABEL[l] in ["Anger", "Disgust"] else 0 for l in labels]

# 🎯 Cải tiến Ngưỡng: P(Anger) + P(Disgust) > 0.4 ➜ Gán cờ 'held' kiểm duyệt
toxic_probs = probs[:, anger_idx] + probs[:, disgust_idx]
y_pred_toxic = [1 if p > 0.4 else 0 for p in toxic_probs]

print("=== BÁO CÁO KIỂM DUYỆT BÌNH LUẬN ĐỘC HẠI CHO HUTUBE (NGƯỠNG P > 0.4) ===")
print(classification_report(
    y_true_toxic,
    y_pred_toxic,
    target_names=["An toàn (Visible)", "Độc hại / Cần kiểm duyệt (Held)"],
    digits=4
))

In [ ]:
# Bước 9: 📊 Vẽ Sơ Đồ Ma Trận Nhầm Lẫn (Confusion Matrix Heatmaps)
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))

cm_labels_vi = [f"{e}\n({EMOTION_VI[e].split('/')[0].strip()})" for e in EMOTION_LABELS]
cm_emotion = confusion_matrix(labels, preds)
sns.heatmap(cm_emotion, annot=True, fmt='d', cmap='Blues', 
            xticklabels=cm_labels_vi, yticklabels=cm_labels_vi, ax=axes[0])
axes[0].set_title("Ma Trận 7 Cảm Xúc (ViSoBERT + Loss Căn Bậc Hai)", fontsize=13, fontweight="bold", pad=10)
axes[0].set_xlabel("Dự Đoán", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Thực Tế", fontsize=11, fontweight="bold")

cm_toxic = confusion_matrix(y_true_toxic, y_pred_toxic)
sns.heatmap(cm_toxic, annot=True, fmt='d', cmap='Reds', 
            xticklabels=["An toàn (Visible)", "Độc hại (Held)"], 
            yticklabels=["An toàn (Visible)", "Độc hại (Held)"], ax=axes[1])
axes[1].set_title("Ma Trận Kiểm Duyệt Độc Hại HuTube (Ngưỡng P > 0.4)", fontsize=13, fontweight="bold", pad=10)
axes[1].set_xlabel("Dự Đoán HuTube", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Thực Tế", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("evaluation_visobert_moderation.png", dpi=300)
plt.show()

In [ ]:
# Bước 10: Thử nghiệm thực tế với Teencode & Emoji mạng xã hội
demo_comments = [
    "Video hướng dẫn siêu có tâm, cảm ơn anh nhiều ạ! ❤️👍",
    "Video xàm vcl lừa đảo quááá, cút đi ad ngu ngốc 😡🤬",
    "Kênh làm chán quá, thất vọng ghê haizzz",
    "Cho mik xin tên bài nhạc ở phút thứ 2 với bạn ơi",
    "Trời ơi xem đoạn cuối bất ngờ thực sự ko tin nổi luôn 😱"
]

model.eval()
print("=== KẾT QUẢ DỰ ĐOÁN THỰC TẾ TRÊN BÌNH LUẬN TEENCODE / EMOJI ===\n")
for cmt in demo_comments:
    clean = clean_social_text(cmt)
    inputs = tokenizer(clean, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
        p = F.softmax(logits, dim=-1).cpu().numpy()[0]
    
    best_idx = np.argmax(p)
    best_emo = ID2LABEL[best_idx]
    score = p[best_idx]
    sentiment_en = EMOTION_TO_SENTIMENT[best_emo]
    sentiment_vi = SENTIMENT_VI[sentiment_en]
    
    # Áp dụng ngưỡng kiểm duyệt: P(Anger) + P(Disgust) > 0.4
    p_toxic = p[anger_idx] + p[disgust_idx]
    if p_toxic > 0.4:
        status_hutube = f"held (GIỮ LẠI KIỂM DUYỆT 🛑 - Độc hại {p_toxic*100:.1f}%)"
    else:
        status_hutube = "visible (CHO PHÉP HIỂN THỊ ✅)"
    
    print(f"💬 Bình luận : '{cmt}'")
    print(f"   ➜ Làm sạch: '{clean}'")
    print(f"   ➜ Cảm xúc : {best_emo} - {EMOTION_VI[best_emo]} ({score*100:.1f}%)")
    print(f"   ➜ Sắc thái: {sentiment_vi} ({sentiment_en})")
    print(f"   ➜ HuTube  : {status_hutube}\n")

In [ ]:
# Bước 11: Nén mô hình đã train và tải về máy để đưa vào emotion-service
SAVE_PATH = "./phobert_vsmec_best"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

!zip -r phobert_vsmec_best.zip phobert_vsmec_best

from google.colab import files
files.download('phobert_vsmec_best.zip')
print("\n[+] Đã tạo file phobert_vsmec_best.zip!")
print("👉 Hãy giải nén file này vào thư mục: emotion-service/saved_models/phobert_vsmec_best/")